# Module 05 — PyTorch `nn.Module` & Optimizers

Modules 02-03 hand-built a network's structure (`Neuron` → `Layer` → `MLP`)
and its training loop (manual loss, manual `.backward()`, manual
`p.data -= lr * p.grad`) on top of the scalar `Value` engine. This module
redoes *exactly the same toy problem* — the same two-blob classification
task — using PyTorch's real building blocks: `nn.Module` for structure and
`torch.optim` for the update rule. Same task, different (production) tool,
so you can see directly what each hand-written piece maps to.

## 1. The same toy dataset as Module 03

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

random.seed(42)


def make_blobs(n_per_class=50):
    X, y = [], []
    centers = [(-1.5, -1.5), (1.5, 1.5)]
    for label, (cx, cy) in enumerate(centers):
        for _ in range(n_per_class):
            X.append([cx + random.gauss(0, 0.6), cy + random.gauss(0, 0.6)])
            y.append(1.0 if label == 0 else -1.0)
    return X, y


Xs_list, ys_list = make_blobs()
X = torch.tensor(Xs_list, dtype=torch.float32)
y = torch.tensor(ys_list, dtype=torch.float32)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Spectral, edgecolors="k")
plt.title("Toy dataset (identical to Module 03)")
plt.show()

## 2. Structure: `nn.Module` instead of hand-rolled `Neuron`/`Layer`/`MLP`

Module 02's `MLP(2, [8, 8, 1])` was three `Layer`s of hand-written
`Neuron`s, each neuron holding its own list of `Value` weights. `nn.Module`
gives us the same shape — a stack of linear layers with nonlinearities in
between — but the weight tensors, forward pass, and parameter bookkeeping
(`.parameters()`) are all handled by the framework.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, hidden_sizes):
        super().__init__()
        sizes = [n_in] + hidden_sizes
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


torch.manual_seed(42)
model = MLP(2, [8, 8, 1])
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"model has {n_params} parameters")

## 3. Loss + optimizer instead of a manual update rule

Module 03 wrote the max-margin hinge loss by hand and added a manual `alpha
* sum(p*p)` L2 term, then did `p.data -= learning_rate * p.grad` for every
parameter itself. Here:
- the hinge loss is the same formula, just as a tensor expression instead
  of `Value` operations
- L2 regularization becomes the optimizer's `weight_decay` argument — no
  manual term needed
- `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` replace
  the manual zero-grad-then-nudge loop

In [ ]:
def hinge_loss(scores, targets):
    return torch.relu(1 - targets * scores).mean()


optimizer = torch.optim.SGD(model.parameters(), lr=1.0, weight_decay=1e-4)

## 4. The training loop

In [ ]:
loss_history = []
for step in range(100):
    scores = model(X)
    loss = hinge_loss(scores, y)

    optimizer.zero_grad()
    loss.backward()

    # match Module 03's decaying learning rate schedule
    lr = 1.0 - 0.9 * step / 100
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.step()

    with torch.no_grad():
        accuracy = ((y > 0) == (scores > 0)).float().mean().item()

    loss_history.append(loss.item())
    if step % 10 == 0:
        print(f"step {step:3d}   loss {loss.item():.4f}   accuracy {accuracy * 100:.1f}%")

assert accuracy == 1.0, "expected the model to reach 100% accuracy on this toy task, same as Module 03"
print("\nFinal accuracy:", accuracy * 100, "%")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training loss over time")
plt.show()

## 5. Seeing what it learned

In [ ]:
import numpy as np

h = 0.25
x_min, x_max = -3.5, 3.5
y_min, y_max = -3.5, 3.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    Z = (model(grid) > 0).float().numpy().reshape(xx.shape)

plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.4)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Spectral, edgecolors="k")
plt.title("Decision boundary (nn.Module + torch.optim)")
plt.show()

## Recap — what mapped to what

| Modules 01-03 (hand-rolled) | Modules 04-05 (PyTorch) |
|---|---|
| `Value` (scalar autograd node) | `torch.Tensor` with `requires_grad=True` |
| `Neuron` / `Layer` / `MLP` classes | `nn.Linear` + `nn.Sequential` inside an `nn.Module` |
| Manual `alpha * sum(p*p)` L2 term | `weight_decay=...` on the optimizer |
| `for p in model.parameters(): p.grad = 0.0` | `optimizer.zero_grad()` |
| `p.data -= learning_rate * p.grad` | `optimizer.step()` |

Same concepts throughout — nothing new conceptually happened here, just a
switch to the tools real training code actually uses. Next up — Module 06:
a counting-based bigram model, the simplest possible language model, as the
starting point for Phase 2's language modeling fundamentals.